# 11 · Actor-Critic
**DEEP RL** · ⏱ 45 min · 🧪 2 exercises · ⭐ 320 XP

*A learned critic turns REINFORCE into an online, low-variance learner — advantages, one-step actor-critic, A2C and GAE(λ).*

> **First time?** Open **00 · START HERE** in the same folder. Then run the cell below to connect this notebook to your Drive folder — your progress and XP are saved there.

In [ ]:
#@title ▶ **Run me first** — connect this notebook to your Reward Signal folder
#@markdown Mounts Google Drive and finds the `RewardSignal` folder (progress, XP and your files live there).
#@markdown Leave `ROOT` empty unless you put the folder somewhere unusual, e.g. `/content/drive/MyDrive/courses/RewardSignal`.
ROOT = ""  #@param {type:"string"}
LESSON = "11-actor-critic"
import os, sys

def _rs_locate(root=""):
    tries = [root, os.environ.get("RS_ROOT", "")]
    try:
        from google.colab import drive  # only exists in Colab
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
        base = "/content/drive/MyDrive"
        tries += [f"{base}/RewardSignal", f"{base}/Colab Notebooks/RewardSignal", f"{base}/RewardSignal/RewardSignal"]
        tries += [os.path.join(base, d, "RewardSignal") for d in sorted(os.listdir(base)) if not d.startswith(".")]
        tries += ["/content/RewardSignal"]
    except ImportError:  # local Jupyter: the folder this notebook sits in
        tries += [os.getcwd(), os.path.dirname(os.getcwd())]
    for t in tries:
        if t and os.path.isfile(os.path.join(t, "runtime", "rlcolab.py")):
            return os.path.abspath(t)
    raise FileNotFoundError("Couldn't find the RewardSignal folder. Upload it to My Drive "
                            "(see its README.md) or type its full path into ROOT, then run this cell again.")

RS_ROOT = _rs_locate(ROOT.strip())
sys.dont_write_bytecode = True  # no __pycache__ folders in your Drive
if os.path.join(RS_ROOT, "runtime") not in sys.path:
    sys.path.insert(0, os.path.join(RS_ROOT, "runtime"))
import rlcolab
rlcolab.setup(RS_ROOT, lesson=LESSON)

REINFORCE has two annoying properties:

- It must wait until the **end of the episode** to compute Gₜ. In a 500-step CartPole episode
  the first action's update has to wait 500 steps; in a continuing task it waits forever.
- Gₜ is a sum of many random rewards, so its **variance** is huge — even with a baseline.

We already know the cure from the tabular chapters: **bootstrap**. Instead of waiting for the
full return, estimate it from one reward plus a learned value of the next state. A policy
gradient method that bootstraps with a learned value function is an **actor-critic**:

- the **actor** π(a | s, θ) chooses actions — trained by policy gradient
- the **critic** v̂(s, w) judges them — trained by TD learning

## 01 · From baseline to critic

Last lesson's baseline v̂(s) was only used to *centre* the return: Gₜ − v̂(Sₜ). An actor-critic
also uses it *inside* the target. The one-step return is Rₜ₊₁ + γ v̂(Sₜ₊₁), so the advantage
estimate becomes the familiar **TD error**:

```text
δₜ = Rₜ₊₁ + γ v̂(Sₜ₊₁, w) − v̂(Sₜ, w)
```

Why is δ an advantage? The **advantage** of an action is how much better it is than the policy's
average from that state:

```text
A_π(s, a) = q_π(s, a) − v_π(s)          and          E[ δₜ | Sₜ = s, Aₜ = a ] = q_π(s, a) − v_π(s)   (if v̂ = v_π)
```

So with an accurate critic, δₜ is an unbiased sample of the advantage — and it involves a single
reward instead of hundreds. With an *inaccurate* critic it is biased. Trading a little bias for a
lot less variance is the bargain at the heart of actor-critic.

## 02 · One-step actor-critic

```python
for each step of each episode:
    a ~ π(·|s, θ);   s2, r, terminated, truncated = env.step(a)
    δ = r + γ (1 − terminated) v̂(s2, w) − v̂(s, w)
    w ← w + α_w δ ∇v̂(s, w)                 # critic: semi-gradient TD(0)
    θ ← θ + α_θ δ ∇log π(a | s, θ)          # actor: policy gradient with δ as the advantage
    s ← s2
```

It is fully **online** — it learns at every step, works for continuing tasks, and needs no
episode memory. Note the two step sizes: the critic usually needs to learn *faster* than the
actor, so the actor's advantages mean something.

In [ ]:
#@title 🧠 **Knowledge check 1** · +25 XP — run, then pick an answer
rlcolab.quiz("11-actor-critic#0", {"q": "In one-step actor-critic, the critic estimates v̂(s) = 10, v̂(s′) = 12, the reward is 1 and γ = 0.9. The action taken will be made…", "options": ["more likely (δ = +1.8)", "less likely (δ = −1.0)", "more likely (δ = +3.0)", "neither — δ = 0"], "answer": 0, "explain": "δ = 1 + 0.9·12 − 10 = 1.8 > 0: things went better than the critic expected, so the actor increases π(a|s)."})

## 03 · n-step returns and GAE(λ)

The one-step TD error is low-variance but leans heavily on the critic. The full Monte Carlo
return is unbiased but noisy. In between are **n-step advantages** — the same spectrum you met
with n-step TD:

```text
Â⁽¹⁾ = δₜ          Â⁽²⁾ = δₜ + γδₜ₊₁          Â⁽ⁿ⁾ = Σₖ₌₀ⁿ⁻¹ γᵏ δₜ₊ₖ          Â⁽∞⁾ = Gₜ − v̂(Sₜ)
```

(The δ's telescope: δₜ + γδₜ₊₁ = Rₜ₊₁ + γRₜ₊₂ + γ²v̂(Sₜ₊₂) − v̂(Sₜ).)

**Generalised Advantage Estimation** (Schulman et al., 2016) does not pick one n — it takes an
exponentially weighted average of all of them, exactly like TD(λ):

```text
Âₜ = Σₖ (γλ)ᵏ δₜ₊ₖ          computed backwards as          Âₜ = δₜ + γλ Âₜ₊₁
```

- **λ = 0** → Âₜ = δₜ (one-step, low variance, biased by the critic)
- **λ = 1** → Âₜ = Gₜ − v̂(Sₜ) (Monte Carlo, unbiased, high variance)
- **λ ≈ 0.95** is the near-universal default.

The critic is trained to predict the **λ-return** Âₜ + v̂(Sₜ), so the same backward pass gives
both the actor's advantages and the critic's targets.

### Worked example

Rollout of three steps, γ = 0.5, λ = 0.5: rewards [1, 0, 2], critic values [1, 2, 1], and the
state after the rollout is worth 4.

- δ₂ = 2 + 0.5·4 − 1 = **3**
- δ₁ = 0 + 0.5·1 − 2 = **−1.5**
- δ₀ = 1 + 0.5·2 − 1 = **1**
- Â₂ = 3,  Â₁ = −1.5 + 0.25·3 = **−0.75**,  Â₀ = 1 + 0.25·(−0.75) = **0.8125**

### Episode boundaries inside a rollout

A rollout of fixed length usually spans several episodes. If step t ended an episode, nothing
after it belongs to the same trajectory: δₜ must not bootstrap (no γv̂(Sₜ₊₁)) and Âₜ must not
accumulate Âₜ₊₁. Multiply both by (1 − doneₜ).

> **⚠️ WARNING** · **Truncation, again**
>
> A time-limit cut is not a real ending — but for GAE it *is* the end of the trajectory in the
> rollout. The standard fix: on truncation, add γ·v̂(s_final) to the last reward, then treat the
> step as done. The provided A2C loop does exactly that.

---
### 🧪 Exercise 1 · Generalised Advantage Estimation · ⭐ **120 XP**

Implement the advantage estimator used by A2C, PPO and most modern policy-gradient code.

- `td_errors(rewards, values, dones, last_value, gamma)` → δₜ = rₜ + γ(1 − doneₜ)V(sₜ₊₁) − V(sₜ). `values[t]` is V(sₜ); the state after the final step has value `last_value`.
- `compute_gae(rewards, values, dones, last_value, gamma, lam)` → `(advantages, returns)` with **Âₜ = δₜ + γλ(1 − doneₜ)Âₜ₊₁**, computed backwards, and **returns = Â + V** (the critic's targets).

A rollout may contain several episodes: `dones[t] = 1` means the episode ended at step t, so neither δ nor Â may look past it. The check includes hand-computed cases and verifies that λ = 0 gives TD errors and λ = 1 gives Monte Carlo returns.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 11-actor-critic/01-gae
import numpy as np
import rlhud


def td_errors(rewards, values, dones, last_value, gamma):
    """One-step TD errors δₜ = rₜ + γ (1 − doneₜ) V(sₜ₊₁) − V(sₜ) for a rollout of length T.

    values[t] = V(sₜ);  the value of the state AFTER the last step is `last_value`.
    dones[t] = 1 if the episode ended at step t (then there is no V(sₜ₊₁) to bootstrap from).
    """
    # TODO: build next_values (values shifted by one, with last_value at the end), then δ
    raise NotImplementedError


def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    """Generalised Advantage Estimation. Returns (advantages, returns), both shape (T,).

    Âₜ = δₜ + γλ (1 − doneₜ) Âₜ₊₁      (computed backwards, Â_T = 0)
    returnsₜ = Âₜ + V(sₜ)             (the critic's regression targets)
    """
    # TODO: walk backwards accumulating Â; reset the accumulation at episode ends
    raise NotImplementedError


if __name__ == "__main__":
    # A 10-step rollout: reward 1 per step, the episode ends at step 5 and a new one starts.
    # The critic is imperfect (it thinks every state is worth 5).
    rewards = np.ones(10)
    values = np.full(10, 5.0)
    dones = np.zeros(10); dones[4] = 1
    for lam in (0.0, 0.5, 0.9, 1.0):
        adv, _ = compute_gae(rewards, values, dones, last_value=5.0, gamma=0.9, lam=lam)
        rlhud.plot(adv, label=f"λ = {lam}", title="GAE advantages along the rollout", xlabel="step t")
    rlhud.text("λ = 0 is the one-step TD error (low variance, biased by the critic); λ = 1 is the "
               "Monte Carlo return minus V (unbiased, high variance). Note how nothing leaks across "
               "the episode boundary after step 4.", title="Reading the chart")

In [ ]:
#@title ✅ **Check** · Generalised Advantage Estimation (+120 XP) — tests the code you last ran above
rlcolab.check("11-actor-critic/01-gae")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

`next_values = np.append(values[1:], last_value)` lines up V(sₜ₊₁) with each step.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

δ = rewards + gamma * (1 - dones) * next_values - values — all vectorised.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

GAE: `running = 0; for t in reversed(range(T)): running = delta[t] + gamma * lam * (1 - dones[t]) * running; adv[t] = running`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("11-actor-critic/01-gae")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 11-actor-critic/01-gae
# Reference solution
import numpy as np
import rlhud


def td_errors(rewards, values, dones, last_value, gamma):
    """One-step TD errors δₜ = rₜ + γ (1 − doneₜ) V(sₜ₊₁) − V(sₜ) for a rollout of length T.

    values[t] = V(sₜ);  the value of the state AFTER the last step is `last_value`.
    dones[t] = 1 if the episode ended at step t (then there is no V(sₜ₊₁) to bootstrap from).
    """
    rewards, values, dones = (np.asarray(v, dtype=float) for v in (rewards, values, dones))
    next_values = np.append(values[1:], last_value)
    return rewards + gamma * (1.0 - dones) * next_values - values


def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    """Generalised Advantage Estimation. Returns (advantages, returns), both shape (T,).

    Âₜ = δₜ + γλ (1 − doneₜ) Âₜ₊₁      (computed backwards, Â_T = 0)
    returnsₜ = Âₜ + V(sₜ)             (the critic's regression targets)
    """
    deltas = td_errors(rewards, values, dones, last_value, gamma)
    dones = np.asarray(dones, dtype=float)
    adv = np.zeros(len(deltas))
    running = 0.0
    for t in reversed(range(len(deltas))):
        running = deltas[t] + gamma * lam * (1.0 - dones[t]) * running
        adv[t] = running
    return adv, adv + np.asarray(values, dtype=float)


if __name__ == "__main__":
    # A 10-step rollout: reward 1 per step, the episode ends at step 5 and a new one starts.
    # The critic is imperfect (it thinks every state is worth 5).
    rewards = np.ones(10)
    values = np.full(10, 5.0)
    dones = np.zeros(10); dones[4] = 1
    for lam in (0.0, 0.5, 0.9, 1.0):
        adv, _ = compute_gae(rewards, values, dones, last_value=5.0, gamma=0.9, lam=lam)
        rlhud.plot(adv, label=f"λ = {lam}", title="GAE advantages along the rollout", xlabel="step t")
    rlhud.text("λ = 0 is the one-step TD error (low variance, biased by the critic); λ = 1 is the "
               "Monte Carlo return minus V (unbiased, high variance). Note how nothing leaks across "
               "the episode boundary after step 4.", title="Reading the chart")

## 04 · A2C: batched actor-critic

Updating on every single transition is noisy and slow in numpy. **A2C** (Advantage Actor-Critic)
collects a short rollout of n steps (often from several environments in parallel), then does one
batched update:

1. run the policy for n steps (e.g. 32), storing states, actions, rewards, dones
2. compute GAE advantages and λ-return targets with the critic
3. **actor loss** L_π = −mean(Âₜ log π(Aₜ|Sₜ)) — normalise Â within the batch first
4. **critic loss** L_V = ½ mean(v̂(Sₜ) − targetₜ)²
5. optionally an **entropy bonus**: subtract β·H(π) from the loss to keep the policy exploring

Writing the objectives as *losses* lets an off-the-shelf minimiser (Adam) do the work: minimising
−Â log π is ascending the policy gradient.

> **ℹ️ NOTE** · **A2C vs A3C**
>
> The original A3C (2016) ran many actor-learners asynchronously on CPU threads. A2C is the
> synchronous version: wait for all environments, update once. It is simpler, deterministic and
> just as good — the "A3" mostly survives in old blog posts.

In this exercise the actor is still a **linear** softmax (enough for CartPole) while the critic
is a small **neural network** — V(s) for CartPole is a curved, symmetric surface that no linear
function of the raw state can represent.

In [ ]:
#@title 🧠 **Knowledge check 2** · +25 XP — run, then pick an answer
rlcolab.quiz("11-actor-critic#1", {"q": "Why normalise the advantages within each batch before the actor update?", "options": ["It is required for the policy gradient theorem to hold", "It keeps the size of the actor's step roughly constant regardless of reward scale and critic accuracy", "It makes the critic learn faster", "It removes the bias of the critic"], "answer": 1, "explain": "Normalisation is a practical trick: it fixes the scale of the update (and centres it). It doesn't remove bias, and the theorem doesn't need it."})

---
### 🧪 Exercise 2 · Advantage actor-critic on CartPole · ⭐ **200 XP**

Train an **A2C** agent: a linear-softmax **actor** and a small neural-network **critic**, updated every 32 steps from GAE advantages.

Implement:
- `compute_gae(...)` — paste your solution from the previous exercise.
- `actor_loss_grad(theta, X, A, adv)` — the gradient w.r.t. θ of the actor **loss** L = −mean(Âₜ log π(Aₜ|Sₜ)). It is the REINFORCE gradient from lesson 10 with a minus sign, because the optimiser minimises.
- `critic_loss_grad(v_pred, targets)` — gradient of ½·mean(v_pred − target)² w.r.t. `v_pred` (shape `(N, 1)`).

The rollout loop, the time-limit bootstrap and both optimisers are provided — read them. The check trains for 250 000 steps (≈ 6 s) and requires a mean return ≥ 150. The demo also plots what the critic thinks V(s) looks like.

> ✏️ **Edit** the next cell and run it (`Shift+Enter`) — that also runs its demo. Then run **✅ Check**. Stuck? The hints are right below the check.

In [ ]:
# ✏️ EXERCISE 11-actor-critic/02-a2c-cartpole
import numpy as np
import rlhud
from rlenvs import CartPole


# ---------------------------------------------------------------- provided: critic network + optimiser
class MLP:
    """Linear → tanh → … → Linear with manual backprop (like lesson 9, tanh instead of ReLU)."""

    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.params = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.params += [rng.normal(0.0, np.sqrt(1.0 / n_in), size=(n_in, n_out)), np.zeros(n_out)]
        self.n_layers = len(sizes) - 1

    def forward(self, x):
        h, self._inputs = np.asarray(x, dtype=float), []
        for i in range(self.n_layers):
            self._inputs.append(h)
            h = h @ self.params[2 * i] + self.params[2 * i + 1]
            if i < self.n_layers - 1:
                h = np.tanh(h)
        return h

    def backward(self, dout):
        grads, d = [None] * len(self.params), np.asarray(dout, dtype=float)
        for i in reversed(range(self.n_layers)):
            grads[2 * i], grads[2 * i + 1] = self._inputs[i].T @ d, d.sum(axis=0)
            if i > 0:
                d = (d @ self.params[2 * i].T) * (1.0 - self._inputs[i] ** 2)   # tanh′ = 1 − tanh²
        return grads


class Adam:
    """Minimises: pass the gradient of a LOSS."""

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            p -= self.lr * (m / (1 - self.b1 ** self.t)) / (np.sqrt(v / (1 - self.b2 ** self.t)) + self.eps)


def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def features(obs):
    return np.append(obs, 1.0)


# ---------------------------------------------------------------- your code
def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    """(advantages, returns) — exactly as in the previous exercise."""
    # TODO: paste your solution from the previous exercise
    raise NotImplementedError


def actor_loss_grad(theta, X, A, adv):
    """Gradient w.r.t. theta of the actor LOSS  L(θ) = −mean_t[ Âₜ log π(Aₜ|Sₜ) ].

    X: (N, d) features, A: (N,) actions, adv: (N,) advantages (treated as constants).
    Minimising this loss = ascending the policy gradient. Returns an array shaped like theta.
    """
    # TODO: the policy gradient from lesson 10, with a minus sign (it is a loss), averaged over N
    raise NotImplementedError


def critic_loss_grad(v_pred, targets):
    """Gradient w.r.t. v_pred (shape (N, 1)) of the critic loss  ½ · mean (v_pred − target)²."""
    # TODO: one line (mind the shapes: v_pred is (N, 1), targets is (N,))
    raise NotImplementedError


# ---------------------------------------------------------------- provided: A2C training loop
def train(total_steps=250_000, seed=0, n_steps=32, gamma=0.99, lam=0.95, lr_actor=5e-3,
          lr_critic=1e-3, hidden=32, value_scale=0.01, log=None):
    """Advantage actor-critic: a linear-softmax actor and a small MLP critic.

    value_scale: the critic network predicts V(s)·value_scale. CartPole values reach ~100;
    networks are happiest with targets around ±1, so we shrink them (and grow them back).
    Returns (theta, critic, episode_returns).
    """
    rng = np.random.default_rng(seed)
    env = CartPole(seed=seed)
    theta = np.zeros((5, env.n_actions))
    critic = MLP([4, hidden, 1], seed=seed)
    actor_opt, critic_opt = Adam([theta], lr=lr_actor), Adam(critic.params, lr=lr_critic)
    V = lambda obs: critic.forward(np.atleast_2d(obs))[:, 0] / value_scale

    obs, _ = env.reset(seed=seed)
    ep_return, returns, t = 0.0, [], 0
    while t < total_steps:
        # 1) collect a short rollout of n_steps with the current policy
        X, OBS, A, R, D = [], [], [], [], []
        for _ in range(n_steps):
            x = features(obs)
            p = softmax(x @ theta)
            a = int(rng.random() < p[1])
            obs2, r, terminated, truncated, _ = env.step(a)
            if truncated and not terminated:
                r += gamma * V(obs2)[0]       # time limit: fold the bootstrap value into r …
            X.append(x); OBS.append(obs); A.append(a); R.append(r)
            D.append(float(terminated or truncated))   # … and then cut the episode like a done
            ep_return += 1.0; t += 1
            obs = obs2
            if terminated or truncated:
                returns.append(ep_return)
                if log:
                    log(len(returns), ep_return)
                ep_return = 0.0
                obs, _ = env.reset()
        X, OBS, A = np.array(X), np.array(OBS), np.array(A)

        # 2) advantages from the critic
        adv, targets = compute_gae(R, V(OBS), D, V(obs)[0], gamma, lam)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        # 3) actor step (policy gradient with the critic's advantage) and critic step (regression)
        actor_opt.step([actor_loss_grad(theta, X, A, adv)])
        v_pred = critic.forward(OBS)
        critic_opt.step(critic.backward(critic_loss_grad(v_pred, targets * value_scale)))
    return theta, critic, returns


def evaluate(theta, episodes=10, seed=1000):
    """Mean return of the stochastic policy."""
    env, rng, total = CartPole(), np.random.default_rng(123), 0.0
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        done = False
        while not done:
            p = softmax(features(obs) @ theta)
            obs, r, term, trunc, _ = env.step(int(rng.random() < p[1]))
            total += r
            done = term or trunc
    return total / episodes


if __name__ == "__main__":
    live = rlhud.chart("A2C on CartPole — episode return", smooth=25, xlabel="episode")
    theta, critic, returns = train(seed=0, log=lambda ep, G: live.add("A2C", G))
    live.flush()
    rlhud.stat("Evaluation (10 episodes)", evaluate(theta))
    # What does the critic believe? V(s) as the pole angle sweeps (cart centred, at rest).
    thetas = np.linspace(-0.2, 0.2, 41)
    vs = critic.forward(np.array([[0.0, 0.0, th, 0.0] for th in thetas]))[:, 0] / 0.01
    rlhud.plot(vs, x=thetas, label="V(s)", title="Critic: value vs pole angle θ", xlabel="θ (rad)")

In [ ]:
#@title ✅ **Check** · Advantage actor-critic on CartPole (+200 XP) — tests the code you last ran above
rlcolab.check("11-actor-critic/02-a2c-cartpole")

<details><summary>💡 <b>Hint 1</b> of 3 — try it before opening the next one</summary>

actor_loss_grad: `P = softmax(X @ theta)`, `onehot = np.eye(theta.shape[1])[A]`.


<details><summary>💡 <b>Hint 2</b> of 3 — try it before opening the next one</summary>

`return -X.T @ ((onehot - P) * adv[:, None]) / len(A)`.


<details><summary>💡 <b>Hint 3</b> of 3 — nearly gives it away</summary>

critic_loss_grad: `return (v_pred - targets.reshape(-1, 1)) / len(v_pred)`.

</details>

</details>

</details>

In [ ]:
#@title 🔒 Reference solution — double-click to reveal · running it replaces your version
rlcolab.solution_gate("11-actor-critic/02-a2c-cartpole")  # obeys the Solutions setting in 00 · START HERE
# ✏️ EXERCISE 11-actor-critic/02-a2c-cartpole
# Reference solution
import numpy as np
import rlhud
from rlenvs import CartPole


# ---------------------------------------------------------------- provided: critic network + optimiser
class MLP:
    """Linear → tanh → … → Linear with manual backprop (like lesson 9, tanh instead of ReLU)."""

    def __init__(self, sizes, seed=0):
        rng = np.random.default_rng(seed)
        self.params = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.params += [rng.normal(0.0, np.sqrt(1.0 / n_in), size=(n_in, n_out)), np.zeros(n_out)]
        self.n_layers = len(sizes) - 1

    def forward(self, x):
        h, self._inputs = np.asarray(x, dtype=float), []
        for i in range(self.n_layers):
            self._inputs.append(h)
            h = h @ self.params[2 * i] + self.params[2 * i + 1]
            if i < self.n_layers - 1:
                h = np.tanh(h)
        return h

    def backward(self, dout):
        grads, d = [None] * len(self.params), np.asarray(dout, dtype=float)
        for i in reversed(range(self.n_layers)):
            grads[2 * i], grads[2 * i + 1] = self._inputs[i].T @ d, d.sum(axis=0)
            if i > 0:
                d = (d @ self.params[2 * i].T) * (1.0 - self._inputs[i] ** 2)   # tanh′ = 1 − tanh²
        return grads


class Adam:
    """Minimises: pass the gradient of a LOSS."""

    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.b1, self.b2, self.eps = params, lr, beta1, beta2, eps
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0

    def step(self, grads):
        self.t += 1
        for p, g, m, v in zip(self.params, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            p -= self.lr * (m / (1 - self.b1 ** self.t)) / (np.sqrt(v / (1 - self.b2 ** self.t)) + self.eps)


def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def features(obs):
    return np.append(obs, 1.0)


# ---------------------------------------------------------------- your code
def compute_gae(rewards, values, dones, last_value, gamma=0.99, lam=0.95):
    """(advantages, returns) — exactly as in the previous exercise."""
    rewards, values, dones = (np.asarray(x, dtype=float) for x in (rewards, values, dones))
    next_values = np.append(values[1:], last_value)
    deltas = rewards + gamma * (1.0 - dones) * next_values - values
    adv, running = np.zeros(len(deltas)), 0.0
    for t in reversed(range(len(deltas))):
        running = deltas[t] + gamma * lam * (1.0 - dones[t]) * running
        adv[t] = running
    return adv, adv + values


def actor_loss_grad(theta, X, A, adv):
    """Gradient w.r.t. theta of the actor LOSS  L(θ) = −mean_t[ Âₜ log π(Aₜ|Sₜ) ].

    X: (N, d) features, A: (N,) actions, adv: (N,) advantages (treated as constants).
    Minimising this loss = ascending the policy gradient. Returns an array shaped like theta.
    """
    P = softmax(X @ theta)
    onehot = np.eye(theta.shape[1])[A]
    return -X.T @ ((onehot - P) * adv[:, None]) / len(A)


def critic_loss_grad(v_pred, targets):
    """Gradient w.r.t. v_pred (shape (N, 1)) of the critic loss  ½ · mean (v_pred − target)²."""
    return (v_pred - np.asarray(targets).reshape(-1, 1)) / len(v_pred)


# ---------------------------------------------------------------- provided: A2C training loop
def train(total_steps=250_000, seed=0, n_steps=32, gamma=0.99, lam=0.95, lr_actor=5e-3,
          lr_critic=1e-3, hidden=32, value_scale=0.01, log=None):
    """Advantage actor-critic: a linear-softmax actor and a small MLP critic.

    value_scale: the critic network predicts V(s)·value_scale. CartPole values reach ~100;
    networks are happiest with targets around ±1, so we shrink them (and grow them back).
    Returns (theta, critic, episode_returns).
    """
    rng = np.random.default_rng(seed)
    env = CartPole(seed=seed)
    theta = np.zeros((5, env.n_actions))
    critic = MLP([4, hidden, 1], seed=seed)
    actor_opt, critic_opt = Adam([theta], lr=lr_actor), Adam(critic.params, lr=lr_critic)
    V = lambda obs: critic.forward(np.atleast_2d(obs))[:, 0] / value_scale

    obs, _ = env.reset(seed=seed)
    ep_return, returns, t = 0.0, [], 0
    while t < total_steps:
        # 1) collect a short rollout of n_steps with the current policy
        X, OBS, A, R, D = [], [], [], [], []
        for _ in range(n_steps):
            x = features(obs)
            p = softmax(x @ theta)
            a = int(rng.random() < p[1])
            obs2, r, terminated, truncated, _ = env.step(a)
            if truncated and not terminated:
                r += gamma * V(obs2)[0]       # time limit: fold the bootstrap value into r …
            X.append(x); OBS.append(obs); A.append(a); R.append(r)
            D.append(float(terminated or truncated))   # … and then cut the episode like a done
            ep_return += 1.0; t += 1
            obs = obs2
            if terminated or truncated:
                returns.append(ep_return)
                if log:
                    log(len(returns), ep_return)
                ep_return = 0.0
                obs, _ = env.reset()
        X, OBS, A = np.array(X), np.array(OBS), np.array(A)

        # 2) advantages from the critic
        adv, targets = compute_gae(R, V(OBS), D, V(obs)[0], gamma, lam)
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        # 3) actor step (policy gradient with the critic's advantage) and critic step (regression)
        actor_opt.step([actor_loss_grad(theta, X, A, adv)])
        v_pred = critic.forward(OBS)
        critic_opt.step(critic.backward(critic_loss_grad(v_pred, targets * value_scale)))
    return theta, critic, returns


def evaluate(theta, episodes=10, seed=1000):
    """Mean return of the stochastic policy."""
    env, rng, total = CartPole(), np.random.default_rng(123), 0.0
    for i in range(episodes):
        obs, _ = env.reset(seed=seed + i)
        done = False
        while not done:
            p = softmax(features(obs) @ theta)
            obs, r, term, trunc, _ = env.step(int(rng.random() < p[1]))
            total += r
            done = term or trunc
    return total / episodes


if __name__ == "__main__":
    live = rlhud.chart("A2C on CartPole — episode return", smooth=25, xlabel="episode")
    theta, critic, returns = train(seed=0, log=lambda ep, G: live.add("A2C", G))
    live.flush()
    rlhud.stat("Evaluation (10 episodes)", evaluate(theta))
    # What does the critic believe? V(s) as the pole angle sweeps (cart centred, at rest).
    thetas = np.linspace(-0.2, 0.2, 41)
    vs = critic.forward(np.array([[0.0, 0.0, th, 0.0] for th in thetas]))[:, 0] / 0.01
    rlhud.plot(vs, x=thetas, label="V(s)", title="Critic: value vs pole angle θ", xlabel="θ (rad)")

> **🔑 KEY IDEA** · **Actor-critic = policy gradient + learned advantage**
>
> The actor follows Â · ∇log π. The critic supplies Â by bootstrapping, which slashes variance
> and allows online, per-rollout updates. GAE's λ dials between the critic's bias (λ → 0) and
> Monte Carlo's variance (λ → 1).

## 05 · Debugging actor-critic

- **Look at the critic first.** Plot the critic loss and a few v̂(s) values. For CartPole with
  γ = 0.99, values should climb toward ~100 as the policy improves. A critic that predicts
  garbage gives the actor garbage advantages.
- **Scale value targets.** Networks like outputs around ±1. Either normalise returns or predict
  V·c for a small c (the `value_scale` in the exercise).
- **Signs.** Adam minimises. Actor loss = −Â·log π; the advantage is a *constant* (no gradient
  flows from the actor loss into the critic).
- **Explained variance** 1 − Var(target − v̂)/Var(target) is a great one-number critic health
  check: near 1 is good, ≤ 0 means the critic is useless.
- **Returns that rise then collapse** often mean the actor's step is too large — the motivation
  for PPO, next.

In [ ]:
#@title 🧠 **Knowledge check 3** · +25 XP — run, then pick an answer
rlcolab.quiz("11-actor-critic#2", {"q": "Your A2C agent's return goes up, then crashes to ~10 and never recovers. Which change is most likely to help?", "options": ["Larger actor learning rate", "Smaller actor learning rate (or a trust-region method like PPO)", "Set λ = 0", "Remove the critic"], "answer": 1, "explain": "A too-large policy step can move the policy somewhere terrible; the new data it collects is then bad too, and it can't recover. Smaller or constrained steps (PPO) fix this."})

---
## 🏁 Debrief
Run the cell below for your mission status, then open the next notebook.

In [ ]:
#@title 🏁 **Mission status** — run to refresh
rlcolab.lesson_footer("11-actor-critic")